# Functional Programming

In [ ]:
import time
from functools import wraps
from contextlib import contextmanager

**Learning goals** — By the end of this section you will be able to:

- Explain **first-class functions** and write **lambda expressions**
- Use `map()`, `filter()`, and `sorted()` with lambdas
- Define and write **higher-order functions**
- Create and apply **decorators** as a form of higher-order function, using `@functools.wraps`
- Build list, dictionary, and set **comprehensions** with conditions
- Write **recursive functions** with a clear base case and recursive case
- Use **context managers** to manage resources safely with the `with` statement

## What is Functional Programming?

**Functional programming (FP)** is a programming paradigm that treats computation as the evaluation of mathematical functions. Rather than describing *how* to do something step-by-step (imperative style), functional programs describe *what* to compute.

Key principles of functional programming:

- **Pure functions** — a function always returns the same output for the same input and has no side effects (it doesn't modify external state)
- **Immutability** — data is not modified in place; new values are produced instead
- **First-class functions** — functions are treated like any other value: they can be assigned to variables, passed as arguments, and returned from other functions
- **Higher-order functions** — functions that accept other functions as arguments or return them (e.g., `map()`, `filter()`, `sorted()`)

Python is a multi-paradigm language; it supports object-oriented, imperative, *and* functional styles. In this chapter we explore Python's functional programming tools: lambda expressions, higher-order functions, decorators, and comprehensions.

## Pure Functions & Immutability

### Pure functions

A **pure function** has two properties:
1. **Deterministic** — given the same inputs, it always returns the same output
2. **No side effects** — it doesn't modify anything outside itself (no global variables, no mutating arguments, no I/O)

Pure functions are easier to test (no setup/teardown needed), easier to reason about, and safe to run in parallel.

### Immutability

**Immutability** means not modifying data in place. Instead of changing an existing object, you produce a *new* object with the desired value. This avoids a whole class of bugs where one part of a program silently changes data that another part is using.

In Python, prefer returning new collections (`lst + [x]`, `{**d, 'key': val}`) over mutating them (`lst.append(x)`, `d['key'] = val`).

In [ ]:
# --- Impure function: depends on external state, has side effects ---
total = 0

def add_to_total(x):       # reads and modifies external variable
    global total
    total += x
    return total

print(add_to_total(5))     # 5
print(add_to_total(5))     # 10  ← same input, different output!

# --- Pure function: same input always gives same output, no side effects ---
def add(a, b):
    return a + b

print(add(5, 3))           # 8
print(add(5, 3))           # 8  ← always the same

# --- Impure: mutates the list passed in ---
def append_zero_impure(lst):
    lst.append(0)          # modifies the original list
    return lst

data = [1, 2, 3]
append_zero_impure(data)
print(data)                # [1, 2, 3, 0]  ← original changed!

# --- Pure: returns a new list, leaves original untouched ---
def append_zero_pure(lst):
    return lst + [0]       # creates a new list

data = [1, 2, 3]
result = append_zero_pure(data)
print(data)                # [1, 2, 3]     ← unchanged
print(result)              # [1, 2, 3, 0]

## First-Class Functions & Lambda

In Python, functions are **first-class values** — they can be assigned to variables, stored in data structures, passed as arguments, and returned from other functions.

**Lambda expressions** create small anonymous functions in a single line. They are most useful as short key or callback functions:

```python
# Syntax: lambda arguments: expression
square = lambda x: x ** 2
add    = lambda x, y: x + y
```

Lambdas are commonly used with `map()`, `filter()`, and `sorted()` — Python's core higher-order functions:

In [ ]:
# Using lambda with map()
numbers = [1, 2, 3, 4, 5]
squared = list(map(lambda x: x ** 2, numbers))
print(squared)  # [1, 4, 9, 16, 25]

# Using lambda with filter()
even_numbers = list(filter(lambda x: x % 2 == 0, numbers))
print(even_numbers)  # [2, 4]

# Using lambda with sorted()
students = [('Alice', 85), ('Bob', 90), ('Charlie', 78)]
sorted_by_grade = sorted(students, key=lambda student: student[1])
print(sorted_by_grade)  # [('Charlie', 78), ('Alice', 85), ('Bob', 90)]

## Higher-Order Functions

A **higher-order function** is a function that either accepts a function as an argument or returns a function as its result. Python's built-in `map()`, `filter()`, and `sorted()` (used above) are all higher-order functions.

You can also write your own:

```python
def apply_twice(f, x):
    return f(f(x))

apply_twice(lambda x: x + 3, 10)  # → 16
```

**Decorators** are a Python idiom built entirely on this idea: a decorator is a higher-order function that takes a function and returns an enhanced version of it.

### Decorators

A **decorator** is a higher-order function — it takes a function as input and returns a modified version. The `@decorator_name` syntax is Python's shorthand for `func = decorator(func)`, making it easy to add cross-cutting behavior (logging, timing, validation) without changing the original function's code.

**How decorators work:**
1. They take a function as input
2. Wrap it in a new function that adds behavior
3. Return the wrapper, applied automatically with `@decorator_name`

### Common Built-in Decorators

- `@property`: Creates getter/setter methods
- `@staticmethod`: Method doesn't need self or cls
- `@classmethod`: Method receives class as first argument
- `@functools.wraps`: Preserves original function metadata

In [ ]:
# Basic decorator example
def my_decorator(func):
    def wrapper():
        print("Something before the function")
        func()
        print("Something after the function")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

say_hello()
# Output:
# Something before the function
# Hello!
# Something after the function

In [1]:
# Class-based decorator
class CountCalls:
    def __init__(self, func):
        self.func = func
        self.count = 0
    
    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"Call #{self.count} of {self.func.__name__}")
        return self.func(*args, **kwargs)

@CountCalls
def say_hi():
    print("Hi there!")

say_hi()
say_hi()
say_hi()

Call #1 of say_hi
Hi there!
Call #2 of say_hi
Hi there!
Call #3 of say_hi
Hi there!


In [ ]:
# Decorator with arguments
def timer_decorator(func):
    import time
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"{func.__name__} took {end_time - start_time:.4f} seconds")
        return result
    return wrapper

@timer_decorator
def slow_function():
    import time
    time.sleep(1)
    return "Done!"

result = slow_function()
print(result)

## List/Dict/Set Comprehensions

Comprehensions provide a concise way to create lists, dictionaries, and sets. They're more Pythonic and often more efficient than traditional loops.

**Advantages of comprehensions:**
- More concise and readable
- Often faster execution
- More Pythonic
- Can be used in-line

**When to use traditional loops:**
- Complex logic that doesn't fit in one line
- Multiple operations per iteration
- Need to break or continue based on conditions
- Debugging complex transformations

In [ ]:
# Basic list comprehension syntax: [expression for item in iterable if condition]

# Traditional loop approach
squares = []
for x in range(10):
    squares.append(x ** 2)
print("Traditional:", squares)

# List comprehension — same result, one line
squares = [x ** 2 for x in range(10)]
print("Comprehension:", squares)

# With a filter condition
even_squares = [x ** 2 for x in range(10) if x % 2 == 0]
print("Even squares:", even_squares)

# Dictionary comprehension
word_lengths = {word: len(word) for word in ["apple", "banana", "cherry"]}
print("Word lengths:", word_lengths)

# Set comprehension (automatically deduplicates)
unique_remainders = {x % 4 for x in range(12)}
print("Unique remainders mod 4:", unique_remainders)

In [ ]:
# Advanced comprehension techniques

# Conditional expression (ternary operator) in comprehension
numbers = range(-5, 6)
absolute_or_zero = [x if x >= 0 else -x for x in numbers]
print("Absolute values:", absolute_or_zero)

# Multiple conditions
filtered_numbers = [x for x in range(20) if x % 2 == 0 if x % 3 == 0]
print("Divisible by 2 and 3:", filtered_numbers)

# Nested dictionary comprehension
students = ['Alice', 'Bob', 'Charlie']
subjects = ['Math', 'Science', 'English']
grades = {student: {subject: 0 for subject in subjects} for student in students}
print("Grade book:", grades)

## Choosing: Comprehension vs `map`/`filter`

Both approaches transform or filter sequences, but they communicate intent differently.

| Situation | Prefer |
|---|---|
| Simple transformation (`x * 2`, `s.upper()`) | Comprehension — reads like English |
| Complex multi-argument function already defined | `map(fn, seq)` — no need to rewrap in a lambda |
| Transforming *and* filtering in one pass | Comprehension — `[f(x) for x in seq if cond(x)]` is cleaner |
| Chaining multiple transformations | `map`/`filter` — function composition can stay readable |
| Result must be lazy / memory-sensitive | `map`/`filter` — they return iterators, not lists |

**Readability rule of thumb**: if the lambda inside `map` or `filter` is longer than ~30 characters, replace it with a named function or a comprehension.

```python
# Hard to read — lambda is too complex
result = list(filter(lambda s: len(s) > 3 and s.startswith('a'), words))

# Clearer as a comprehension
result = [s for s in words if len(s) > 3 and s.startswith('a')]
```

**Pipeline readability check**: a functional pipeline of 2–3 steps is fine. For longer chains, break steps into named intermediate variables so the intent stays obvious.

In [ ]:
### Exercise: Sorting and Filtering with Lambda
#   Given the student records below (name, grade, year):
#   1. Use filter() with a lambda to keep students with grade >= 80.
#   2. Use sorted() with a lambda to rank passing students by grade, highest first.
#   3. Use map() with a lambda to extract just the names.
#   4. Print the final list of names.
### Your code starts here.
students = [
    ('Alice', 85, 'junior'), ('Bob', 72, 'senior'),
    ('Charlie', 91, 'sophomore'), ('Diana', 65, 'junior'), ('Eve', 88, 'senior')
]

### Your code ends here.

In [ ]:
### Solution
students = [
    ('Alice', 85, 'junior'), ('Bob', 72, 'senior'),
    ('Charlie', 91, 'sophomore'), ('Diana', 65, 'junior'), ('Eve', 88, 'senior')
]
passing = list(filter(lambda s: s[1] >= 80, students))
ranked  = sorted(passing, key=lambda s: s[1], reverse=True)
names   = list(map(lambda s: s[0], ranked))
print(names)  # ['Charlie', 'Eve', 'Alice']

## Summary

- **First-class functions** let you pass and return functions like any other value.
- **Lambda expressions** create short anonymous functions for use with `map()`, `filter()`, and `sorted()`.
- **Decorators** are higher-order functions that wrap and extend behavior without modifying the original function.
- **Comprehensions** offer a concise, readable syntax for transforming and filtering sequences.
- Choose comprehensions for readability; prefer `map`/`filter` when working with lazy iterators or pre-defined functions.

Next: recursion, context managers, and `functools` utilities.